In [21]:
import plotly
import numpy as np
import plotly.graph_objs as go

In [ ]:
u_min, u_max = 0.0, 0.8
v_min, v_max = 0.0, 0.8

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)
U, V = np.meshgrid(u, v)


A = 0.1
angle = 0
phase = np.pi/3
frequency = np.pi/0.8


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.5):
    x = u
    y = v

    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.sin(frequency*s + phase) # bump
    return x, y, z

X, Y, Z = gamma_sur(U, V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z)
])

fig.update_layout(
    title="γ(u, v): bump surface",
    scene=dict(
        aspectmode='data',
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)


Opening in existing browser session.


In [ ]:
import sympy as sp

u, v = sp.symbols("u v")

# is the surface developable
theta = sp.pi/2
A = 0.5
phase = sp.pi/6
s = u * sp.cos(theta) + v * sp.sin(theta)
gamma = sp.Matrix([u, v, A*sp.cos(s + phase)])

E = (gamma.diff(u)).dot(gamma.diff(u))
F = gamma.diff(u).dot(gamma.diff(v))
G = (gamma.diff(v)).dot(gamma.diff(v))
L = gamma.diff(u, u)
M = gamma.diff(u, v)
N = gamma.diff(v, v)

n = gamma.diff(u).cross(gamma.diff(v))
a = sp.sqrt(n.dot(n))
n = n/a

L = L.dot(n)
M = M.dot(n)
N = N.dot(n)

print((L * N - M**2) / (E * G - F**2))

# if it is zero, then the surface is developable

0


In [1]:
# consider a fully actuated sheet (i.e., all the joints are controllable)
import pinocchio as pin

model = pin.buildModelFromSdf("mass_mesh_open_tree.sdf", root_link_name="mass_x0_y0")
# data = model.createData()

: 

In [1]:
# print(len(constraints))
print(len(model.joints))
print(model.nv)

NameError: name 'model' is not defined

In [4]:
frame_ids = dict()

num_elements_x = 5
num_elements_y = 5

origin_x_ind = int(num_elements_x / 2)
origin_y_ind = int(num_elements_y / 2)

for i in range(num_elements_x):
    for j in range(num_elements_y):
        frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

In [ ]:
q = pin.neutral(model)
pin.forwardKinematics(model, data, q)
link_positions = []
for i in range(num_elements_x):
    for j in range(num_elements_y):
        pin.updateFramePlacement(model, data, frame_ids[(i, j)])
        print(f"Frame mass_x{i}_y{j} placement:\n{data.oMf[frame_ids[(i, j)]].translation}\n")
        link_positions.append(data.oMf[frame_ids[(i, j)]].translation)

: 

In [80]:
u_min, u_max = 0.0, 0.8
v_min, v_max = 0.0, 0.8

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)

goals_u = np.linspace(u_min, u_max, num_elements_x)
goals_v = np.linspace(v_min, v_max, num_elements_y)

U, V = np.meshgrid(u, v)
goals_U, goals_V = np.meshgrid(goals_u, goals_v)

A = 0.1
angle = 0
phase = 0
frequency = np.pi/0.8


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.5):
    x = u
    y = v

    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.sin(frequency*s + phase) # bump
    return x, y, z

X, Y, Z = gamma_sur(U, V, A=A)
goals_X, goals_Y, goals_Z = gamma_sur(goals_U, goals_V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z),
    go.Scatter3d(x=goals_X.flatten(), y=goals_Y.flatten(), z=goals_Z.flatten(), 
            mode='markers', marker=dict(size=23, color='red')),
    go.Scatter3d(x=[pos[0] for pos in link_positions],y=[pos[1] for pos in link_positions],z=[pos[2] for pos in link_positions],
            mode='markers', marker=dict(size=12, color='blue'))
])

fig.update_layout(
    title="γ(u, v): bump surface",
    scene=dict(
        aspectmode='data',
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)


Opening in existing browser session.
